# Land Cover Model Training
**Dataset:** EuroSAT All Bands (Sentinel-2, 13-band, 64×64 patches)  
**Link:** https://www.kaggle.com/datasets/apollo2506/eurosat-dataset  
**Output:** `land_cover_model.pkl`  
**Model:** XGBoost Classifier  
**Approach:** Extract true NDVI (B8/B4) + spectral features per patch → classify dominant land cover  
**Input features at prediction time:** `ndvi`, `slope_deg`, `elevation`

In [ ]:
import numpy as np
import pandas as pd
import joblib
import rasterio
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

BASE        = '/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSATallBands'
MAX_PATCHES = 20000  # 20k is enough, full dataset is ~27k

## 1. Class Mapping
EuroSAT 10 classes → our 5 platform classes

In [ ]:
# EuroSAT label → our platform class
LABEL_MAP = {
    0: 'cropland',    # AnnualCrop
    1: 'vegetation',  # Forest
    2: 'vegetation',  # HerbaceousVegetation
    3: 'urban',       # Highway
    4: 'urban',       # Industrial
    5: 'vegetation',  # Pasture
    6: 'cropland',    # PermanentCrop
    7: 'urban',       # Residential
    8: 'water',       # River
    9: 'water',       # SeaLake
}

# Fixed class order — must match services/land_cover.py CLASSES exactly
CLASSES     = ['vegetation', 'urban', 'barren', 'water', 'cropland']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
print('Class → ID mapping:', CLASS_TO_ID)

## 2. Load CSV Index

In [ ]:
train_df = pd.read_csv(f'{BASE}/train.csv')
val_df   = pd.read_csv(f'{BASE}/validation.csv')
df       = pd.concat([train_df, val_df], ignore_index=True)

# Map to our classes
df['our_class'] = df['Label'].map(LABEL_MAP)
df['class_id']  = df['our_class'].map(CLASS_TO_ID)

print(f'Total patches: {len(df)}')
print('\nClass distribution:')
print(df['our_class'].value_counts())

## 3. Feature Extraction

Sentinel-2 band layout (13 bands, uint16, range 0–10000 reflectance):
- **B4 = index 3** (Red) → NDVI denominator
- **B8 = index 7** (NIR) → NDVI numerator  
- **B3 = index 2** (Green) → vegetation/water signal
- **B11 = index 11** (SWIR1) → elevation/moisture proxy
- **B12 = index 12** (SWIR2) → barren/urban signal

Per-patch features extracted → summary stats that approximate `ndvi`, `slope_deg`, `elevation`

In [ ]:
def extract_features(filepath):
    with rasterio.open(filepath) as f:
        bands = f.read().astype(np.float32)  # shape: (13, 64, 64)

    # Normalize reflectance to 0-1 (Sentinel-2 uint16 max = 10000)
    bands = bands / 10000.0

    B3  = bands[2]   # Green
    B4  = bands[3]   # Red
    B8  = bands[7]   # NIR
    B11 = bands[11]  # SWIR1
    B12 = bands[12]  # SWIR2

    # True NDVI = (NIR - Red) / (NIR + Red)
    ndvi = (B8 - B4) / (B8 + B4 + 1e-8)

    # NDWI (water index) = (Green - NIR) / (Green + NIR)
    ndwi = (B3 - B8) / (B3 + B8 + 1e-8)

    # NDBI (built-up index) = (SWIR1 - NIR) / (SWIR1 + NIR)
    ndbi = (B11 - B8) / (B11 + B8 + 1e-8)

    # BSI (bare soil index) = ((SWIR1 + Red) - (NIR + Blue)) / ((SWIR1 + Red) + (NIR + Blue))
    B2  = bands[1]  # Blue
    bsi = ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2) + 1e-8)

    return [
        float(ndvi.mean()),    # true NDVI mean → maps to our ndvi feature
        float(ndvi.std()),     # NDVI variance — mixed vs uniform land
        float(ndwi.mean()),    # water index — high = water
        float(ndbi.mean()),    # built-up index — high = urban
        float(bsi.mean()),     # bare soil index — high = barren/cropland
        float(B11.mean()),     # SWIR1 mean → elevation/moisture proxy
        float(B11.std()),      # SWIR1 variance → slope proxy
        float(B8.mean()),      # NIR mean — vegetation density
        float(B4.mean()),      # Red mean — soil/urban reflectance
        float(B3.mean()),      # Green mean
    ]

In [ ]:
# Sample MAX_PATCHES with stratification to keep class balance
df_sampled = (
    df.groupby('our_class', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MAX_PATCHES // df['our_class'].nunique()), random_state=42))
    .reset_index(drop=True)
)
print(f'Sampled patches: {len(df_sampled)}')
print(df_sampled['our_class'].value_counts())

In [ ]:
all_features, all_labels = [], []

for _, row in tqdm(df_sampled.iterrows(), total=len(df_sampled)):
    filepath = f"{BASE}/{row['Filename']}"
    try:
        feats = extract_features(filepath)
        all_features.append(feats)
        all_labels.append(row['class_id'])
    except Exception:
        continue

X = np.array(all_features)
y = np.array(all_labels)

print(f'Feature matrix: {X.shape}')
print('Class distribution:')
unique, counts = np.unique(y, return_counts=True)
for uid, cnt in zip(unique, counts):
    print(f'  {CLASSES[uid]}: {cnt}')

## 4. Train Full Spectral Model (XGBoost on 10 features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
spectral_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
spectral_model.fit(X_train, y_train,
                   eval_set=[(X_test, y_test)],
                   verbose=50)

y_pred = spectral_model.predict(X_test)
print(f'\nSpectral model accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
FEATURE_NAMES = ['ndvi_mean', 'ndvi_std', 'ndwi_mean', 'ndbi_mean', 'bsi_mean',
                 'swir1_mean', 'swir1_std', 'nir_mean', 'red_mean', 'green_mean']
print('Feature importances:')
for name, imp in sorted(zip(FEATURE_NAMES, spectral_model.feature_importances_),
                         key=lambda x: -x[1]):
    print(f'  {name}: {imp:.4f}')

## 5. Bridge Model (Knowledge Distillation → 3 features)

Our platform calls `predict_land_cover(ndvi, slope_deg, elevation)` at prediction time.
We distill the spectral model's knowledge into a 3-feature bridge model:
- `ndvi` → directly from `ndvi_mean` (true NDVI this time)
- `slope_deg` → from `swir1_std` (SWIR variance correlates with terrain roughness)
- `elevation` → from `swir1_mean` (SWIR1 reflectance correlates with elevation/dryness)

In [ ]:
# Build 3-feature matrix matching platform inputs
# ndvi_mean=X[:,0], swir1_std=X[:,6], swir1_mean=X[:,5]
X_bridge = np.column_stack([
    X[:, 0],          # ndvi_mean  → ndvi
    X[:, 6] * 100,    # swir1_std  → slope_deg proxy (scale to degrees range)
    X[:, 5] * 3000,   # swir1_mean → elevation proxy (scale to meters range)
])

# Use spectral model's soft probability outputs as distillation targets
# Soft labels carry more information than hard class predictions
y_soft = spectral_model.predict_proba(X)  # shape: (N, 5)
y_hard = spectral_model.predict(X)        # hard labels for accuracy eval

X_br_train, X_br_test, y_br_train, y_br_test, y_hard_train, y_hard_test = train_test_split(
    X_bridge, y_soft, y_hard, test_size=0.2, random_state=42, stratify=y
)
print(f'Bridge train: {X_br_train.shape}, test: {X_br_test.shape}')

In [ ]:
# Train one bridge classifier per class using soft probability as target
# Then combine into a single model via XGBoost multi-output
# Simpler: train XGBoost directly on hard labels from spectral model
bridge_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
bridge_model.fit(
    X_br_train, y_hard_train,
    eval_set=[(X_br_test, y_hard_test)],
    verbose=100
)

br_pred = bridge_model.predict(X_br_test)
print(f'\nBridge model accuracy: {accuracy_score(y_hard_test, br_pred):.4f}')
print(classification_report(y_hard_test, br_pred, target_names=CLASSES))

## 6. Save Model

In [ ]:
joblib.dump(bridge_model, 'land_cover_model.pkl')
print('Saved: land_cover_model.pkl')

# Verify — inputs must match services/land_cover.py: ndvi, slope_deg, elevation
loaded = joblib.load('land_cover_model.pkl')

test_cases = [
    [0.72,  2.0,  145.0],  # high NDVI, gentle slope  → vegetation
    [0.05,  1.5,   38.0],  # low NDVI, flat, low elev → urban or water
    [0.08,  8.0,  420.0],  # low NDVI, steep, high    → barren
    [0.35,  3.0,  200.0],  # moderate NDVI            → cropland
    [-0.05, 0.5,   10.0],  # negative NDVI            → water
]

print('\nVerification:')
for tc in test_cases:
    pred_id   = int(loaded.predict([tc])[0])
    pred_class = CLASSES[pred_id]
    proba     = loaded.predict_proba([tc])[0]
    print(f'ndvi={tc[0]:5.2f}, slope={tc[1]:4.1f}, elev={tc[2]:6.1f} '
          f'→ {pred_class:12s} {dict(zip(CLASSES, proba.round(2)))}')